# Reward ablation, stage 1: feasibility runs
Trains three fresh adapters from the same Qwen2.5-1.5B base for 20 steps each, seed 3407. Only the correctness reward differs:

- `historical`: the pilot's original answer parser (rejects `<answer>4</answer>.`)
- `corrected`: the fixed parser used for evaluation
- `format_only`: no correctness reward at all, as a control

Everything else matches the pilot: the same 128 training questions, prompt, LoRA, GRPO settings and model revision. See `docs/NEXT_EXPERIMENT.md` for the full design.

Stage 1 checks that each arm trains properly (every step logged, nonzero gradients, all adapter tensors changed) and records time and memory. It does not evaluate on the reserved 256 questions, and these adapters are not used later.

1. Open a T4 GPU session, run the installation cell, then select **Runtime → Restart session**.
2. Skip installation and run the remaining cells in order.
3. Each arm takes roughly 15 minutes. If the session disconnects, reconnect and run the cells again (skipping installation): finished arms are skipped and an interrupted arm restarts from scratch.

Outputs go to **My Drive → aha_reward_ablation → stage1**.


In [ ]:
%pip install "transformers==4.56.2" "trl==0.22.2" "peft==0.17.1" "accelerate==1.10.1" "datasets==4.1.1" "bitsandbytes==0.48.1"
# Restart session now, then skip this cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json
from datetime import datetime, timezone
import torch
assert torch.cuda.is_available(), 'Choose Runtime > Change runtime type > T4 GPU'
print(torch.cuda.get_device_name(0))
REPO_URL='https://github.com/dch2r/Chasing-the-Aha-Moment'
ROOT=Path('/content/Chasing-the-Aha-Moment')
if not ROOT.exists():
    subprocess.run(['git','clone','--depth','1',REPO_URL,str(ROOT)],check=True)
COMMIT=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD']).decode().strip()
subprocess.run([sys.executable,str(ROOT/'scripts/verify_results.py')],check=True)
OUT=Path('/content/drive/MyDrive/aha_reward_ablation/stage1')
OUT.mkdir(parents=True,exist_ok=True)
print('Repository commit:',COMMIT)
print('Outputs:',OUT)

In [ ]:
# Rerun this cell after a disconnect; finished arms are skipped.
SEED=3407
def finished(run):
    p=run/'diagnostics_summary.json'
    return p.exists() and json.loads(p.read_text()).get('complete') is True
for arm in ['historical','corrected','format_only']:
    run=OUT/f'{arm}_seed{SEED}'
    if finished(run):
        print('Already finished:',arm)
        continue
    if run.exists():
        stamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        run.rename(run.with_name(f'{run.name}_incomplete_{stamp}'))
        print('Kept the interrupted attempt aside and restarting:',arm)
    print(f'=== Training {arm} ===',flush=True)
    log=OUT/f'{arm}_seed{SEED}.log'
    with log.open('a') as f:
        f.write(f'commit {COMMIT}\n')
        p=subprocess.Popen([sys.executable,str(ROOT/'scripts/train_arm.py'),'--arm',arm,
            '--seed',str(SEED),'--steps','20','--output',str(run)],
            stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout:
            print(line,end='');f.write(line)
        p.wait()
    assert p.returncode==0,f'{arm} failed; see {log}'
    torch.cuda.empty_cache()
print('All three arms finished.')

In [ ]:
summary=OUT/'stage1_summary.json'
cmd=[sys.executable,str(ROOT/'scripts/summarize_ablation.py'),str(OUT),'--seed','3407']
if not summary.exists():
    cmd+=['--output',str(summary)]
subprocess.run(cmd)
print('Share this folder when done:',OUT)